In [56]:
import pandas as pd
df = pd.read_csv('Chocolate Sales (2).csv')

# Display the first few rows of the DataFrame, summary information, statistical description, and check for missing values to have a better understanding of the dataset.
# This will help us identify any potential issues or insights before performing further analysis.

print(df.head())
print(df.info())    
print(df.describe())
print(df.isnull().sum())



     Sales Person    Country              Product        Date      Amount  \
0  Jehu Rudeforth         UK      Mint Chip Choco  04/01/2022   $5,320.00   
1     Van Tuxwell      India        85% Dark Bars  01/08/2022   $7,896.00   
2    Gigi Bohling      India  Peanut Butter Cubes  07/07/2022   $4,501.00   
3    Jan Morforth  Australia  Peanut Butter Cubes  27/04/2022  $12,726.00   
4  Jehu Rudeforth         UK  Peanut Butter Cubes  24/02/2022  $13,685.00   

   Boxes Shipped  
0            180  
1             94  
2             91  
3            342  
4            184  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3282 entries, 0 to 3281
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Sales Person   3282 non-null   object
 1   Country        3282 non-null   object
 2   Product        3282 non-null   object
 3   Date           3282 non-null   object
 4   Amount         3282 non-null   object
 5   Boxes Shipp

In [57]:
# Rename columns to be more easily accessible in code and to follow Python naming conventions (lowercase with underscores).

from pandas import Index


df = df.rename(columns={
    "Sales Person": "sales_person",
    "Country": "country",
    "Product": "product",
    "Date": "date",
    "Amount": "amount",
    "Boxes Shipped": "boxes_shipped"
})

Index(['sales_person', 'country', 'product', 'date', 'amount', 'boxes_shipped'], dtype='object')


print(df.columns)


Index(['sales_person', 'country', 'product', 'date', 'amount',
       'boxes_shipped'],
      dtype='object')


In [58]:
# Remove any leading or trailing whitespace from the column names to ensure they are clean and consistent.
text_cols = ["sales_person", "country", "product"]

for col in text_cols:
 df[col] = df[col].str.strip()

# Text uniformity
df["sales_person"] = df["sales_person"].str.title()
df["country"] = df["country"].str.title()
df["product"] = df["product"].str.title()

# Amount column: Remove any currency symbols and convert the values to numeric type for analysis.
df["amount"] = (
 df["amount"]
 .str.replace("$", "", regex=False)
 .str.replace(",", "", regex=False)
 .str.strip()
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

In [59]:
# Convert the 'date' column to datetime format to facilitate date-related operations and analyses.
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce")
print(df["date"].dtype)
print(df["date"].head())
print(df["date"].isnull().sum())

datetime64[ns]
0   2022-01-04
1   2022-08-01
2   2022-07-07
3   2022-04-27
4   2022-02-24
Name: date, dtype: datetime64[ns]
0


In [60]:
# ensure that the 'boxes_shipped' column is of numeric type, which will allow for accurate calculations and analyses related to the number of boxes shipped.
df["boxes_shipped"] = pd.to_numeric(df["boxes_shipped"], errors="coerce")

In [61]:
# missing values
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
sales_person     0
country          0
product          0
date             0
amount           0
boxes_shipped    0
dtype: int64


In [62]:
#remove duplicate rows to ensure that the dataset is clean and does not contain any redundant entries that could skew analysis results.
df = df.drop_duplicates()
print(f"Number of rows after removing duplicates: {len(df)}")
print("Duplicate rows:", df.duplicated().sum())

Number of rows after removing duplicates: 3282
Duplicate rows: 0


In [63]:
#remove invalid entries in the 'amount' and 'boxes_shipped' columns, such as negative values or zeros, which are not meaningful in the context of sales data and could distort analysis results.
df = df[df["amount"] > 0] 
df = df[df["boxes_shipped"] > 0]

In [64]:
#create new columns for future analysis, such as 'year' and 'month' extracted from the 'date' column, which can help in identifying trends and patterns over time.
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["year_month"] = df["date"].dt.to_period("M").astype(str)
df["amount_per_box"] = df["amount"] / df["boxes_shipped"]

print("Data Info:")
print(df.info())

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3282 entries, 0 to 3281
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   sales_person    3282 non-null   object        
 1   country         3282 non-null   object        
 2   product         3282 non-null   object        
 3   date            3282 non-null   datetime64[ns]
 4   amount          3282 non-null   float64       
 5   boxes_shipped   3282 non-null   int64         
 6   year            3282 non-null   int32         
 7   month           3282 non-null   int32         
 8   month_name      3282 non-null   object        
 9   year_month      3282 non-null   object        
 10  amount_per_box  3282 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int32(2), int64(1), object(5)
memory usage: 256.5+ KB
None


In [ ]:
#Data summary
#Separately summarize the numeric columns and the date column to provide insights into the distribution of sales amounts, boxes shipped, and the range of dates in the dataset.

print("Numeric Summary:")
print(df[["amount", "boxes_shipped", "amount_per_box"]].describe())

print("\nDate Summary:")
print("Earliest date:", df["date"].min().date())
print("Latest date:", df["date"].max().date())
print("Number of unique dates:", df["date"].nunique())

Numeric Summary:
             amount  boxes_shipped  amount_per_box
count   3282.000000    3282.000000     3282.000000
mean    6030.338775     164.666971      111.340158
std     4393.980200     124.024736      295.314947
min        7.000000       1.000000        0.013514
25%     2521.495000      71.000000       15.460976
50%     5225.500000     137.000000       38.192954
75%     8556.842500     232.000000       83.808388
max    26170.950000     778.000000     4692.360000

Date Summary:
Earliest date: 2022-01-03
Latest date: 2024-08-31
Number of unique dates: 504


In [74]:
df.to_csv("chocolate_sales_clean.csv", index=False)
print("File exported successfully.")

File exported successfully.
